# V6 Step 8: Evaluation, EER, Multi-seed Summary

This notebook evaluates the existing V6 Step 7 RP-only RanPAC outputs. It does not rerun Random Projection or RanPAC, does not include no-RP baselines, and does not create dim1000 experiments.


In [6]:
from pathlib import Path
import json
import math
import re

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

ROOT = Path(r"F:\ECG")
STEP7_SUMMARY_PATH = ROOT / "data" / "processed" / "ranpac_v6" / "ranpac_v6_summary.csv"
OUTPUT_DIR = ROOT / "results" / "eval_v6"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
EER_CURVE_DIR = OUTPUT_DIR / "eer_curves"
REPORT_DIR = OUTPUT_DIR / "reports"

FINAL_TABLE_PATH = TABLE_DIR / "final_v6_135.csv"
MEAN_STD_PATH = TABLE_DIR / "mean_std_v6.csv"
REPORT_PATH = REPORT_DIR / "v6_report.md"
NOTEBOOK_PATH = ROOT / "notebooks" / "08_v6_eval.ipynb"

EXPECTED_INPUT_ROWS = 135
EXPECTED_MEAN_STD_ROWS = 45
EXPECTED_NUM_SEEDS = 3
STABILITY_DOWNSAMPLE_STEP = 100

for folder in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, EER_CURVE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

REQUIRED_SCORE_ARRAYS = [
    "true_labels",
    "predicted_labels",
    "correct_flags",
    "seen_class_before_flags",
    "true_scores",
    "max_non_true_scores",
    "strict_counted_flags",
    "enrollment_aware_counted_flags",
]

print("Notebook path =", NOTEBOOK_PATH)
print("Input Step 7 summary =", STEP7_SUMMARY_PATH)
print("Output folder =", OUTPUT_DIR)


Notebook path = F:\ECG\notebooks\08_v6_eval.ipynb
Input Step 7 summary = F:\ECG\data\processed\ranpac_v6\ranpac_v6_summary.csv
Output folder = F:\ECG\results\eval_v6


## 1. Validate V6 Step 7 inputs

Step 8 starts from the V6 Step 7 summary. It must contain exactly 135 RP-only rows, with no no-RP baseline rows.


In [9]:
def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def as_bool_series(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    })


require(STEP7_SUMMARY_PATH.exists(), f"Missing V6 Step 7 summary: {STEP7_SUMMARY_PATH}")
step7_df = pd.read_csv(STEP7_SUMMARY_PATH)
input_step7_rows = int(len(step7_df))

is_rp_flags = as_bool_series(step7_df["is_rp"])
is_baseline_flags = as_bool_series(step7_df["is_baseline"])
no_norp_input = bool(
    is_rp_flags.all()
    and (~is_baseline_flags).all()
    and (step7_df["rp_alias"].astype(str) != "norp").all()
    and (step7_df["seed_label"].astype(str) != "norp").all()
)

score_outputs_exist_input = bool(step7_df["score_output_path"].map(lambda path: Path(str(path)).exists()).all())
prediction_logs_exist_input = bool(step7_df["prediction_log_path"].map(lambda path: Path(str(path)).exists()).all())

require(input_step7_rows == EXPECTED_INPUT_ROWS, f"Expected 135 V6 Step 7 rows, got {input_step7_rows}")
require(no_norp_input, "V6 Step 7 summary includes no-RP or baseline rows")
require(score_outputs_exist_input, "Some score outputs are missing")
require(prediction_logs_exist_input, "Some prediction logs are missing")

print("Input Step 7 rows =", input_step7_rows)
print("RP-only rows =", int(is_rp_flags.sum()))
print("No no-RP rows in input =", no_norp_input)
print("Score outputs exist =", score_outputs_exist_input)
print("Prediction logs exist =", prediction_logs_exist_input)


Input Step 7 rows = 135
RP-only rows = 135
No no-RP rows in input = True
Score outputs exist = True
Prediction logs exist = True


## 2. EER helper functions

EER uses genuine score = true_score and impostor score = max_non_true_score. NaN/Inf scores are filtered because early stream samples can have no previously seen class.


In [12]:
def compute_eer_curve(genuine_scores, impostor_scores):
    genuine_scores = np.asarray(genuine_scores, dtype=float)
    impostor_scores = np.asarray(impostor_scores, dtype=float)
    valid_mask = np.isfinite(genuine_scores) & np.isfinite(impostor_scores)
    genuine_scores = genuine_scores[valid_mask]
    impostor_scores = impostor_scores[valid_mask]

    if len(genuine_scores) == 0 or len(impostor_scores) == 0:
        empty_curve = pd.DataFrame(columns=["threshold", "FAR", "FRR", "abs_FAR_minus_FRR"])
        return {
            "eer": np.nan,
            "threshold": np.nan,
            "curve": empty_curve,
            "num_samples": 0,
        }

    all_scores = np.concatenate([genuine_scores, impostor_scores])
    min_score = float(np.min(all_scores))
    max_score = float(np.max(all_scores))
    score_span = max(max_score - min_score, 1.0)
    epsilon = score_span * 1e-6
    thresholds = np.unique(np.concatenate([[min_score - epsilon], all_scores, [max_score + epsilon]]))
    thresholds.sort()

    genuine_sorted = np.sort(genuine_scores)
    impostor_sorted = np.sort(impostor_scores)

    frr = np.searchsorted(genuine_sorted, thresholds, side="left") / len(genuine_sorted)
    far = (len(impostor_sorted) - np.searchsorted(impostor_sorted, thresholds, side="left")) / len(impostor_sorted)
    abs_diff = np.abs(far - frr)
    best_index = int(np.argmin(abs_diff))

    curve = pd.DataFrame({
        "threshold": thresholds,
        "FAR": far,
        "FRR": frr,
        "abs_FAR_minus_FRR": abs_diff,
    })

    return {
        "eer": float((far[best_index] + frr[best_index]) / 2.0),
        "threshold": float(thresholds[best_index]),
        "curve": curve,
        "num_samples": int(len(genuine_scores)),
    }


def clean_name(value):
    text = str(value)
    text = re.sub(r"[^A-Za-z0-9_\-]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")


## 3. Compute EER and save final per-experiment table

This cell computes strict valid EER and enrollment-aware EER for all 135 experiments, then saves `final_v6_135.csv`.


In [15]:
final_rows = []

for _, row in step7_df.iterrows():
    score_path = Path(str(row["score_output_path"]))
    score_data = np.load(score_path, allow_pickle=False)
    missing_arrays = [name for name in REQUIRED_SCORE_ARRAYS if name not in score_data.files]
    require(not missing_arrays, f"Missing score arrays in {score_path}: {missing_arrays}")

    true_scores = score_data["true_scores"].astype(float)
    max_non_true_scores = score_data["max_non_true_scores"].astype(float)
    enrollment_flags = score_data["enrollment_aware_counted_flags"].astype(bool)

    valid_score_mask = np.isfinite(true_scores) & np.isfinite(max_non_true_scores)
    enrollment_score_mask = valid_score_mask & enrollment_flags

    strict_eer_result = compute_eer_curve(true_scores[valid_score_mask], max_non_true_scores[valid_score_mask])
    enrollment_eer_result = compute_eer_curve(true_scores[enrollment_score_mask], max_non_true_scores[enrollment_score_mask])

    final_rows.append({
        "experiment_id": row["experiment_id"],
        "feature_setting_full": row["feature_setting_full"],
        "feature_alias": row["feature_alias"],
        "rp_method_full": row["rp_method_full"],
        "rp_alias": row["rp_alias"],
        "projection_dim": int(row["projection_dim"]),
        "output_dim": int(row["output_dim"]),
        "seed": int(row["seed"]),
        "seed_label": row["seed_label"],
        "strict_prequential_accuracy": float(row["strict_prequential_accuracy"]),
        "enrollment_aware_accuracy": float(row["enrollment_aware_accuracy"]),
        "strict_macro_f1": float(row["strict_macro_f1"]),
        "enrollment_aware_macro_f1": float(row["enrollment_aware_macro_f1"]),
        "strict_valid_eer": strict_eer_result["eer"],
        "enrollment_aware_eer": enrollment_eer_result["eer"],
        "strict_eer_threshold": strict_eer_result["threshold"],
        "enrollment_aware_eer_threshold": enrollment_eer_result["threshold"],
        "num_valid_score_samples": int(valid_score_mask.sum()),
        "num_enrollment_aware_score_samples": int(enrollment_score_mask.sum()),
        "runtime_seconds": float(row["runtime_seconds"]),
        "prediction_log_path": row["prediction_log_path"],
        "score_output_path": row["score_output_path"],
    })

final_df = pd.DataFrame(final_rows)
require(len(final_df) == EXPECTED_INPUT_ROWS, f"Expected 135 final rows, got {len(final_df)}")
require(final_df[["strict_valid_eer", "enrollment_aware_eer"]].notna().all().all(), "Some EER values were not computed")
final_df.to_csv(FINAL_TABLE_PATH, index=False, encoding="utf-8-sig")

print("Final per-experiment rows =", len(final_df))
print("EER computed for all 135 =", final_df[["strict_valid_eer", "enrollment_aware_eer"]].notna().all().all())
print("Saved final table =", FINAL_TABLE_PATH)


Final per-experiment rows = 135
EER computed for all 135 = True
Saved final table = F:\ECG\results\eval_v6\tables\final_v6_135.csv


## 4. Multi-seed mean/std summary and rankings

The 135 per-experiment rows are aggregated into 45 unique feature/method/dimension settings, each with three seeds.


In [18]:
group_cols = [
    "feature_setting_full",
    "feature_alias",
    "rp_method_full",
    "rp_alias",
    "projection_dim",
]

mean_std_v6 = (
    final_df
    .groupby(group_cols, as_index=False)
    .agg(
        mean_enrollment_aware_accuracy=("enrollment_aware_accuracy", "mean"),
        std_enrollment_aware_accuracy=("enrollment_aware_accuracy", "std"),
        mean_enrollment_aware_macro_f1=("enrollment_aware_macro_f1", "mean"),
        std_enrollment_aware_macro_f1=("enrollment_aware_macro_f1", "std"),
        mean_enrollment_aware_eer=("enrollment_aware_eer", "mean"),
        std_enrollment_aware_eer=("enrollment_aware_eer", "std"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        std_runtime_seconds=("runtime_seconds", "std"),
        num_seeds=("seed", "nunique"),
    )
    .sort_values(group_cols)
    .reset_index(drop=True)
)

require(len(mean_std_v6) == EXPECTED_MEAN_STD_ROWS, f"Expected 45 mean/std rows, got {len(mean_std_v6)}")
require((mean_std_v6["num_seeds"] == EXPECTED_NUM_SEEDS).all(), "Not every mean/std row has 3 seeds")

mean_std_v6["rank_by_mean_accuracy"] = mean_std_v6["mean_enrollment_aware_accuracy"].rank(ascending=False, method="min").astype(int)
mean_std_v6["rank_by_mean_macro_f1"] = mean_std_v6["mean_enrollment_aware_macro_f1"].rank(ascending=False, method="min").astype(int)
mean_std_v6["rank_by_mean_eer"] = mean_std_v6["mean_enrollment_aware_eer"].rank(ascending=True, method="min").astype(int)
mean_std_v6["overall_rank_score"] = (
    mean_std_v6["rank_by_mean_accuracy"]
    + mean_std_v6["rank_by_mean_macro_f1"]
    + mean_std_v6["rank_by_mean_eer"]
)

mean_std_v6.to_csv(MEAN_STD_PATH, index=False, encoding="utf-8-sig")

ranking_tables = {
    "top10_acc.csv": mean_std_v6.sort_values(
        ["mean_enrollment_aware_accuracy", "mean_enrollment_aware_macro_f1", "mean_enrollment_aware_eer"],
        ascending=[False, False, True],
    ).head(10),
    "top10_f1.csv": mean_std_v6.sort_values(
        ["mean_enrollment_aware_macro_f1", "mean_enrollment_aware_accuracy", "mean_enrollment_aware_eer"],
        ascending=[False, False, True],
    ).head(10),
    "top10_eer.csv": mean_std_v6.sort_values(
        ["mean_enrollment_aware_eer", "mean_enrollment_aware_accuracy", "mean_enrollment_aware_macro_f1"],
        ascending=[True, False, False],
    ).head(10),
    "top10_balanced.csv": mean_std_v6.sort_values(
        ["overall_rank_score", "rank_by_mean_accuracy", "rank_by_mean_eer"],
        ascending=[True, True, True],
    ).head(10),
}

for file_name, table in ranking_tables.items():
    table.to_csv(TABLE_DIR / file_name, index=False, encoding="utf-8-sig")

print("Mean/std rows =", len(mean_std_v6))
print("Every mean/std row has num_seeds = 3 =", bool((mean_std_v6["num_seeds"] == 3).all()))
print("Ranking tables saved =", all((TABLE_DIR / name).exists() for name in ranking_tables))
print("Best balanced setting:")
print(ranking_tables["top10_balanced.csv"].head(1).to_string(index=False))


Mean/std rows = 45
Every mean/std row has num_seeds = 3 = True
Ranking tables saved = True
Best balanced setting:
feature_setting_full feature_alias     rp_method_full rp_alias  projection_dim  mean_enrollment_aware_accuracy  std_enrollment_aware_accuracy  mean_enrollment_aware_macro_f1  std_enrollment_aware_macro_f1  mean_enrollment_aware_eer  std_enrollment_aware_eer  mean_runtime_seconds  std_runtime_seconds  num_seeds  rank_by_mean_accuracy  rank_by_mean_macro_f1  rank_by_mean_eer  overall_rank_score
  embedding_resnet1d           emb bernoulli_gaussian       bg             500                        0.818863                       0.000803                        0.785632                       0.001296                    0.21421                  0.000584             41.339193             4.177451          3                      1                      1                 1                   3


## 5. Group-level comparison tables

Group-level comparisons are computed from the 45-row mean/std table, not from the 135 raw seed rows.


In [21]:
GROUP_METRICS = [
    "mean_enrollment_aware_accuracy",
    "mean_enrollment_aware_macro_f1",
    "mean_enrollment_aware_eer",
    "mean_runtime_seconds",
]

def make_group_table(data, group_col):
    rows = []
    for group_value, group_data in data.groupby(group_col, dropna=False):
        out = {group_col: group_value}
        for metric in GROUP_METRICS:
            values = group_data[metric].astype(float)
            out[f"{metric}_mean"] = float(values.mean())
            out[f"{metric}_std"] = float(values.std(ddof=1))
            if metric in ["mean_enrollment_aware_eer", "mean_runtime_seconds"]:
                out[f"{metric}_best"] = float(values.min())
                out[f"{metric}_worst"] = float(values.max())
            else:
                out[f"{metric}_best"] = float(values.max())
                out[f"{metric}_worst"] = float(values.min())
        rows.append(out)
    return pd.DataFrame(rows)

by_feature = make_group_table(mean_std_v6, "feature_setting_full")
by_method = make_group_table(mean_std_v6, "rp_method_full")
by_dim = make_group_table(mean_std_v6, "projection_dim")

by_feature.to_csv(TABLE_DIR / "by_feature.csv", index=False, encoding="utf-8-sig")
by_method.to_csv(TABLE_DIR / "by_method.csv", index=False, encoding="utf-8-sig")
by_dim.to_csv(TABLE_DIR / "by_dim.csv", index=False, encoding="utf-8-sig")

group_table_paths = [TABLE_DIR / "by_feature.csv", TABLE_DIR / "by_method.csv", TABLE_DIR / "by_dim.csv"]
print("Group tables saved =", all(path.exists() for path in group_table_paths))


Group tables saved = True


## 6. Plot helper functions

Figures are generated as PNG files under `results/eval_v6/figures`, separate from all V5 figures.


In [73]:
def load_font(size=18, bold=False):
    candidates = []
    if bold:
        candidates += [r"C:\Windows\Fonts\arialbd.ttf", r"C:\Windows\Fonts\calibrib.ttf"]
    candidates += [r"C:\Windows\Fonts\arial.ttf", r"C:\Windows\Fonts\calibri.ttf"]
    for candidate in candidates:
        try:
            if Path(candidate).exists():
                return ImageFont.truetype(candidate, size=size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_TITLE = load_font(26, bold=True)
FONT_AXIS = load_font(18)
FONT_SMALL = load_font(14)
FONT_TINY = load_font(12)

FONT_TITLE_CLEAR = load_font(34, bold=True)
FONT_AXIS_CLEAR = load_font(24)
FONT_TICK_CLEAR = load_font(24)
FONT_VALUE_CLEAR = load_font(24, bold=True)
FONT_LABEL_CLEAR = load_font(24)

COLORS = {
    "blue": (52, 111, 173),
    "orange": (226, 132, 45),
    "green": (72, 153, 92),
    "red": (194, 77, 72),
    "purple": (120, 95, 170),
    "gray": (95, 95, 95),
    "light_gray": (232, 235, 240),
    "dark": (30, 35, 42),
    "white": (255, 255, 255),
}
PALETTE = [COLORS["blue"], COLORS["orange"], COLORS["green"], COLORS["red"], COLORS["purple"], (70, 150, 160), (160, 110, 60)]


def label_text(row):
    return f"{row['feature_alias']}_{row['rp_alias']}_d{int(row['projection_dim'])}"


def draw_centered(draw, xy, text, font, fill):
    x, y = xy
    box = draw.textbbox((0, 0), str(text), font=font)
    draw.text((x - (box[2] - box[0]) / 2, y), str(text), font=font, fill=fill)


def save_bar_chart(table, category_col, value_col, output_path, title, x_label, y_label, color):
    table = table.copy()
    values = table[value_col].astype(float).to_numpy()
    labels = table[category_col].astype(str).tolist()
    width, height = 1120, 760
    plot_left, plot_right, plot_top, plot_bottom = 120, width - 70, 105, height - 175
    plot_width, plot_height = plot_right - plot_left, plot_bottom - plot_top
    image = Image.new("RGB", (width, height), COLORS["white"])
    draw = ImageDraw.Draw(image)
    draw_centered(draw, (width / 2, 28), title, FONT_TITLE, COLORS["dark"])
    draw_centered(draw, (width / 2, height - 48), x_label, FONT_AXIS, COLORS["dark"])
    draw.text((18, 34), y_label, font=FONT_AXIS, fill=COLORS["dark"])

    y_min = min(0.0, float(values.min()) * 0.95)
    y_max = max(float(values.max()) * 1.12, 0.01)
    if y_max <= 1.0:
        y_max = min(1.0, max(y_max, 0.1))

    for i in range(6):
        frac = i / 5
        y = plot_bottom - frac * plot_height
        draw.line((plot_left, y, plot_right, y), fill=COLORS["light_gray"], width=1)
        tick_value = y_min + frac * (y_max - y_min)
        draw.text((plot_left - 92, y - 8), f"{tick_value:.3f}", font=FONT_TINY, fill=COLORS["gray"])
    draw.line((plot_left, plot_top, plot_left, plot_bottom), fill=COLORS["dark"], width=2)
    draw.line((plot_left, plot_bottom, plot_right, plot_bottom), fill=COLORS["dark"], width=2)

    slot_width = plot_width / max(len(values), 1)
    bar_width = slot_width * 0.58
    for i, (label, value) in enumerate(zip(labels, values)):
        x_center = plot_left + slot_width * (i + 0.5)
        x0, x1 = x_center - bar_width / 2, x_center + bar_width / 2
        y0 = plot_bottom - (value - y_min) / (y_max - y_min) * plot_height
        draw.rectangle((x0, y0, x1, plot_bottom), fill=color)
        draw.text((x0, y0 - 20), f"{value:.3f}", font=FONT_TINY, fill=COLORS["dark"])
        label = str(label)
        if len(label) > 18:
            label = label[:18]
        draw_centered(draw, (x_center, plot_bottom + 14), label, FONT_TINY, COLORS["dark"])
    image.save(output_path)


def draw_multiline_centered(draw, xy, text, font, fill, spacing=4):
    x, y = xy
    lines = str(text).split("\n")
    widths = []
    heights = []
    for line in lines:
        box = draw.textbbox((0, 0), line, font=font)
        widths.append(box[2] - box[0])
        heights.append(box[3] - box[1])
    current_y = y
    for line, width, height in zip(lines, widths, heights):
        draw.text((x - width / 2, current_y), line, font=font, fill=fill)
        current_y += height + spacing


def pretty_rp_label(label, multiline=False):
    raw = str(label).strip().lower()
    mapping = {
        "bernoulli_gaussian": "Bernoulli Gaussian",
        "gaussian": "Gaussian",
        "sparse": "Sparse",
        "sign": "Sign",
        "srht": "SRHT",
    }
    clean = mapping.get(raw, str(label).replace("_", " ").title())
    if multiline and clean == "Bernoulli Gaussian":
        return "Bernoulli\nGaussian"
    return clean


def save_bar_chart_clear(table, category_col, value_col, output_path, title, x_label, y_label, color):
    table = table.copy()
    values = table[value_col].astype(float).to_numpy()
    labels = [pretty_rp_label(label, multiline=True) for label in table[category_col].astype(str).tolist()]

    width, height = 1320, 900
    plot_left, plot_right, plot_top, plot_bottom = 140, width - 60, 120, height - 230
    plot_width, plot_height = plot_right - plot_left, plot_bottom - plot_top

    image = Image.new("RGB", (width, height), COLORS["white"])
    draw = ImageDraw.Draw(image)

    draw_centered(draw, (width / 2, 30), title, FONT_TITLE_CLEAR, COLORS["dark"])
    draw_centered(draw, (width / 2, height - 62), x_label, FONT_AXIS_CLEAR, COLORS["dark"])
    draw.text((20, 40), y_label, font=FONT_AXIS_CLEAR, fill=COLORS["dark"])

    y_min = 0.0
    y_max = max(float(values.max()) * 1.12, 0.10)
    if y_max <= 1.0:
        y_max = min(1.0, y_max)

    for i in range(6):
        frac = i / 5
        y = plot_bottom - frac * plot_height
        draw.line((plot_left, y, plot_right, y), fill=COLORS["light_gray"], width=1)
        tick_value = y_min + frac * (y_max - y_min)
        draw.text((plot_left - 88, y - 10), f"{tick_value:.3f}", font=FONT_TICK_CLEAR, fill=COLORS["gray"])

    draw.line((plot_left, plot_top, plot_left, plot_bottom), fill=COLORS["dark"], width=2)
    draw.line((plot_left, plot_bottom, plot_right, plot_bottom), fill=COLORS["dark"], width=2)

    slot_width = plot_width / max(len(values), 1)
    bar_width = slot_width * 0.58

    for i, (label, value) in enumerate(zip(labels, values)):
        x_center = plot_left + slot_width * (i + 0.5)
        x0, x1 = x_center - bar_width / 2, x_center + bar_width / 2
        y0 = plot_bottom - (value - y_min) / (y_max - y_min) * plot_height

        draw.rectangle((x0, y0, x1, plot_bottom), fill=color)

        value_text = f"{value:.3f}"
        value_box = draw.textbbox((0, 0), value_text, font=FONT_VALUE_CLEAR)
        value_w = value_box[2] - value_box[0]
        value_h = value_box[3] - value_box[1]
        value_y = max(plot_top + 4, y0 - value_h - 8)
        draw.text((x_center - value_w / 2, value_y), value_text, font=FONT_VALUE_CLEAR, fill=COLORS["dark"])

        draw_multiline_centered(
            draw,
            (x_center, plot_bottom + 16),
            label,
            FONT_LABEL_CLEAR,
            COLORS["dark"],
            spacing=4,
        )

    image.save(output_path)


def save_scatter(data, output_path):
    width, height = 1120, 760
    plot_left, plot_right, plot_top, plot_bottom = 115, width - 190, 100, height - 100
    plot_width, plot_height = plot_right - plot_left, plot_bottom - plot_top
    image = Image.new("RGB", (width, height), COLORS["white"])
    draw = ImageDraw.Draw(image)
    draw_centered(draw, (width / 2, 28), "Mean Accuracy vs Mean EER", FONT_TITLE, COLORS["dark"])
    draw_centered(draw, ((plot_left + plot_right) / 2, height - 54), "Mean enrollment-aware EER (lower is better)", FONT_AXIS, COLORS["dark"])
    draw.text((16, 32), "Mean enrollment-aware accuracy", font=FONT_AXIS, fill=COLORS["dark"])

    x_values = data["mean_enrollment_aware_eer"].astype(float).to_numpy()
    y_values = data["mean_enrollment_aware_accuracy"].astype(float).to_numpy()
    x_min, x_max = float(x_values.min()), float(x_values.max())
    y_min, y_max = float(y_values.min()), float(y_values.max())
    x_pad = max((x_max - x_min) * 0.08, 1e-4)
    y_pad = max((y_max - y_min) * 0.08, 1e-4)
    x_min, x_max, y_min, y_max = x_min - x_pad, x_max + x_pad, y_min - y_pad, y_max + y_pad

    for i in range(6):
        frac = i / 5
        x = plot_left + frac * plot_width
        y = plot_bottom - frac * plot_height
        draw.line((x, plot_top, x, plot_bottom), fill=COLORS["light_gray"], width=1)
        draw.line((plot_left, y, plot_right, y), fill=COLORS["light_gray"], width=1)
        draw.text((x - 22, plot_bottom + 10), f"{x_min + frac * (x_max - x_min):.3f}", font=FONT_TINY, fill=COLORS["gray"])
        draw.text((plot_left - 78, y - 8), f"{y_min + frac * (y_max - y_min):.3f}", font=FONT_TINY, fill=COLORS["gray"])
    draw.line((plot_left, plot_top, plot_left, plot_bottom), fill=COLORS["dark"], width=2)
    draw.line((plot_left, plot_bottom, plot_right, plot_bottom), fill=COLORS["dark"], width=2)

    feature_colors = {
        "fid": COLORS["orange"],
        "emb": COLORS["blue"],
        "both": COLORS["green"],
    }
    for _, row in data.iterrows():
        x = plot_left + (row["mean_enrollment_aware_eer"] - x_min) / (x_max - x_min) * plot_width
        y = plot_bottom - (row["mean_enrollment_aware_accuracy"] - y_min) / (y_max - y_min) * plot_height
        color = feature_colors.get(str(row["feature_alias"]), COLORS["purple"])
        draw.ellipse((x - 5, y - 5, x + 5, y + 5), fill=color, outline=COLORS["white"], width=1)

    legend_x, legend_y = plot_right + 24, plot_top + 10
    for label, color in feature_colors.items():
        draw.rectangle((legend_x, legend_y, legend_x + 14, legend_y + 14), fill=color)
        draw.text((legend_x + 22, legend_y - 2), label, font=FONT_SMALL, fill=COLORS["dark"])
        legend_y += 26
    image.save(output_path)


def save_top10_dual_bar(data, output_path):
    top10 = data.sort_values(["overall_rank_score", "rank_by_mean_accuracy", "rank_by_mean_eer"]).head(10).copy()
    width, height = 1260, 820
    plot_left, plot_right, plot_top, plot_bottom = 400, width - 70, 95, height - 80
    plot_width = plot_right - plot_left
    row_height = (plot_bottom - plot_top) / len(top10)
    image = Image.new("RGB", (width, height), COLORS["white"])
    draw = ImageDraw.Draw(image)
    draw_centered(draw, (width / 2, 28), "Balanced Top 10: Mean Accuracy and EER", FONT_TITLE, COLORS["dark"])
    draw_centered(draw, ((plot_left + plot_right) / 2, height - 46), "Metric value", FONT_AXIS, COLORS["dark"])
    for i in range(6):
        frac = i / 5
        x = plot_left + frac * plot_width
        draw.line((x, plot_top, x, plot_bottom), fill=COLORS["light_gray"], width=1)
        draw.text((x - 14, plot_bottom + 10), f"{frac:.1f}", font=FONT_TINY, fill=COLORS["gray"])
    draw.rectangle((plot_left, plot_top, plot_right, plot_bottom), outline=COLORS["dark"], width=2)
    for idx, (_, row) in enumerate(top10.iterrows()):
        y_center = plot_top + row_height * (idx + 0.5)
        label = f"{idx + 1}. {label_text(row)}"
        draw.text((24, y_center - 13), label[:54], font=FONT_TINY, fill=COLORS["dark"])
        acc = float(row["mean_enrollment_aware_accuracy"])
        eer = float(row["mean_enrollment_aware_eer"])
        draw.rectangle((plot_left, y_center - 17, plot_left + acc * plot_width, y_center - 4), fill=COLORS["blue"])
        draw.rectangle((plot_left, y_center + 4, plot_left + eer * plot_width, y_center + 17), fill=COLORS["orange"])
        draw.text((plot_left + acc * plot_width + 6, y_center - 19), f"acc {acc:.3f}", font=FONT_TINY, fill=COLORS["dark"])
        draw.text((plot_left + eer * plot_width + 6, y_center + 2), f"eer {eer:.3f}", font=FONT_TINY, fill=COLORS["dark"])
    image.save(output_path)


def save_error_line(table, x_col, mean_col, std_col, output_path, title, y_label, color, lower_is_better=False):
    table = table.sort_values(x_col).copy()
    x_values = table[x_col].astype(float).to_numpy()
    means = table[mean_col].astype(float).to_numpy()
    stds = table[std_col].astype(float).to_numpy()
    width, height = 1000, 720
    plot_left, plot_right, plot_top, plot_bottom = 105, width - 70, 90, height - 95
    plot_width, plot_height = plot_right - plot_left, plot_bottom - plot_top
    image = Image.new("RGB", (width, height), COLORS["white"])
    draw = ImageDraw.Draw(image)
    draw_centered(draw, (width / 2, 25), title, FONT_TITLE, COLORS["dark"])
    draw_centered(draw, ((plot_left + plot_right) / 2, height - 50), "Projection dimension", FONT_AXIS, COLORS["dark"])
    draw.text((18, 30), y_label, font=FONT_AXIS, fill=COLORS["dark"])
    y_min = float(np.min(means - stds)) * 0.95
    y_max = float(np.max(means + stds)) * 1.05
    if y_max == y_min:
        y_max += 1.0
    for i in range(6):
        frac = i / 5
        y = plot_bottom - frac * plot_height
        draw.line((plot_left, y, plot_right, y), fill=COLORS["light_gray"], width=1)
        draw.text((plot_left - 76, y - 8), f"{y_min + frac * (y_max - y_min):.3f}", font=FONT_TINY, fill=COLORS["gray"])
    draw.line((plot_left, plot_top, plot_left, plot_bottom), fill=COLORS["dark"], width=2)
    draw.line((plot_left, plot_bottom, plot_right, plot_bottom), fill=COLORS["dark"], width=2)
    x_min, x_max = float(x_values.min()), float(x_values.max())
    if x_max == x_min:
        x_max += 1.0
    points = []
    for x_value, mean, std in zip(x_values, means, stds):
        x = plot_left + (x_value - x_min) / (x_max - x_min) * plot_width
        y = plot_bottom - (mean - y_min) / (y_max - y_min) * plot_height
        y_low = plot_bottom - (mean - std - y_min) / (y_max - y_min) * plot_height
        y_high = plot_bottom - (mean + std - y_min) / (y_max - y_min) * plot_height
        draw.line((x, y_low, x, y_high), fill=color, width=2)
        draw.line((x - 6, y_low, x + 6, y_low), fill=color, width=2)
        draw.line((x - 6, y_high, x + 6, y_high), fill=color, width=2)
        draw.ellipse((x - 5, y - 5, x + 5, y + 5), fill=color)
        draw_centered(draw, (x, plot_bottom + 12), str(int(x_value)), FONT_TINY, COLORS["dark"])
        points.append((x, y))
    if len(points) >= 2:
        draw.line(points, fill=color, width=3)
    image.save(output_path)


def save_multi_line(curves, output_path, title, x_label, y_label, y_min=None, y_max=None):
    width, height = 1260, 820
    plot_left, plot_right, plot_top, plot_bottom = 110, width - 230, 95, height - 95
    plot_width, plot_height = plot_right - plot_left, plot_bottom - plot_top
    image = Image.new("RGB", (width, height), COLORS["white"])
    draw = ImageDraw.Draw(image)
    draw_centered(draw, (width / 2, 28), title, FONT_TITLE, COLORS["dark"])
    draw_centered(draw, ((plot_left + plot_right) / 2, height - 54), x_label, FONT_AXIS, COLORS["dark"])
    draw.text((18, 32), y_label, font=FONT_AXIS, fill=COLORS["dark"])

    all_x = np.concatenate([np.asarray(curve["x"], dtype=float) for curve in curves if len(curve["x"]) > 0])
    all_y = np.concatenate([np.asarray(curve["y"], dtype=float) for curve in curves if len(curve["y"]) > 0])
    finite = np.isfinite(all_y)
    all_x = all_x[np.isfinite(all_x)]
    all_y = all_y[finite]
    x_min, x_max = float(np.min(all_x)), float(np.max(all_x))
    if y_min is None:
        y_min = float(np.nanmin(all_y))
    if y_max is None:
        y_max = float(np.nanmax(all_y))
    y_pad = max((y_max - y_min) * 0.08, 1e-4)
    y_min, y_max = y_min - y_pad, y_max + y_pad
    if x_max == x_min:
        x_max += 1.0
    if y_max == y_min:
        y_max += 1.0

    for i in range(6):
        frac = i / 5
        x = plot_left + frac * plot_width
        y = plot_bottom - frac * plot_height
        draw.line((x, plot_top, x, plot_bottom), fill=COLORS["light_gray"], width=1)
        draw.line((plot_left, y, plot_right, y), fill=COLORS["light_gray"], width=1)
        draw.text((x - 24, plot_bottom + 10), f"{x_min + frac * (x_max - x_min):.0f}", font=FONT_TINY, fill=COLORS["gray"])
        draw.text((plot_left - 74, y - 8), f"{y_min + frac * (y_max - y_min):.3f}", font=FONT_TINY, fill=COLORS["gray"])
    draw.line((plot_left, plot_top, plot_left, plot_bottom), fill=COLORS["dark"], width=2)
    draw.line((plot_left, plot_bottom, plot_right, plot_bottom), fill=COLORS["dark"], width=2)

    legend_x, legend_y = plot_right + 24, plot_top + 8
    for idx, curve in enumerate(curves):
        x_arr = np.asarray(curve["x"], dtype=float)
        y_arr = np.asarray(curve["y"], dtype=float)
        mask = np.isfinite(x_arr) & np.isfinite(y_arr)
        x_arr = x_arr[mask]
        y_arr = y_arr[mask]
        color = PALETTE[idx % len(PALETTE)]
        points = [
            (
                plot_left + (x - x_min) / (x_max - x_min) * plot_width,
                plot_bottom - (y - y_min) / (y_max - y_min) * plot_height,
            )
            for x, y in zip(x_arr, y_arr)
        ]
        if len(points) >= 2:
            draw.line(points, fill=color, width=3)
        draw.line((legend_x, legend_y, legend_x + 30, legend_y), fill=color, width=4)
        draw.text((legend_x + 38, legend_y - 8), str(curve["label"])[:26], font=FONT_TINY, fill=COLORS["dark"])
        legend_y += 24
    image.save(output_path)


## 7. Generate required V6 figures

Figures summarize group-level results, rankings, dimension stability, and online learning stability.


In [76]:
by_feature_plot = by_feature[["feature_setting_full", "mean_enrollment_aware_accuracy_mean"]].rename(
    columns={"mean_enrollment_aware_accuracy_mean": "value"}
)
save_bar_chart(by_feature_plot, "feature_setting_full", "value", FIGURE_DIR / "acc_by_feature.png", "Mean Accuracy by Feature", "Feature setting", "Mean accuracy", COLORS["blue"])

by_method_plot = by_method[["rp_method_full", "mean_enrollment_aware_accuracy_mean"]].rename(
    columns={"mean_enrollment_aware_accuracy_mean": "value"}
)
save_bar_chart_clear(by_method_plot, "rp_method_full", "value", FIGURE_DIR / "acc_by_method.png", "Mean Accuracy by RP Method", "RP method", "Mean accuracy", COLORS["green"])

by_dim_plot = by_dim[["projection_dim", "mean_enrollment_aware_accuracy_mean"]].rename(
    columns={"mean_enrollment_aware_accuracy_mean": "value"}
)
save_bar_chart(by_dim_plot, "projection_dim", "value", FIGURE_DIR / "acc_by_dim.png", "Mean Accuracy by Projection Dimension", "Projection dimension", "Mean accuracy", COLORS["purple"])

by_feature_eer_plot = by_feature[["feature_setting_full", "mean_enrollment_aware_eer_mean"]].rename(
    columns={"mean_enrollment_aware_eer_mean": "value"}
)
save_bar_chart(by_feature_eer_plot, "feature_setting_full", "value", FIGURE_DIR / "eer_by_feature.png", "Mean EER by Feature", "Feature setting", "Mean EER", COLORS["orange"])

by_method_eer_plot = by_method[["rp_method_full", "mean_enrollment_aware_eer_mean"]].rename(
    columns={"mean_enrollment_aware_eer_mean": "value"}
)
save_bar_chart_clear(by_method_eer_plot, "rp_method_full", "value", FIGURE_DIR / "eer_by_method.png", "Mean EER by RP Method", "RP method", "Mean EER", COLORS["red"])

by_dim_eer_plot = by_dim[["projection_dim", "mean_enrollment_aware_eer_mean"]].rename(
    columns={"mean_enrollment_aware_eer_mean": "value"}
)
save_bar_chart(by_dim_eer_plot, "projection_dim", "value", FIGURE_DIR / "eer_by_dim.png", "Mean EER by Projection Dimension", "Projection dimension", "Mean EER", COLORS["orange"])

save_scatter(mean_std_v6, FIGURE_DIR / "acc_eer_scatter.png")
save_top10_dual_bar(mean_std_v6, FIGURE_DIR / "top10_acc_eer.png")

dim_std_table = (
    mean_std_v6
    .groupby("projection_dim", as_index=False)
    .agg(
        mean_acc=("mean_enrollment_aware_accuracy", "mean"),
        std_acc=("mean_enrollment_aware_accuracy", "std"),
        mean_eer=("mean_enrollment_aware_eer", "mean"),
        std_eer=("mean_enrollment_aware_eer", "std"),
    )
)
save_error_line(dim_std_table, "projection_dim", "mean_acc", "std_acc", FIGURE_DIR / "acc_dim_std.png", "Accuracy Mean +/- Std by Dimension", "Mean accuracy", COLORS["blue"])
save_error_line(dim_std_table, "projection_dim", "mean_eer", "std_eer", FIGURE_DIR / "eer_dim_std.png", "EER Mean +/- Std by Dimension", "Mean EER", COLORS["orange"])

required_figure_names = [
    "acc_by_feature.png",
    "acc_by_method.png",
    "acc_by_dim.png",
    "eer_by_feature.png",
    "eer_by_method.png",
    "eer_by_dim.png",
    "acc_eer_scatter.png",
    "top10_acc_eer.png",
    "acc_dim_std.png",
    "eer_dim_std.png",
]
print("Base figures saved =", all((FIGURE_DIR / name).exists() for name in required_figure_names))


Base figures saved = True


## 8. Additional RP-focused figures and controlled comparison table for thesis

This section adds thesis-oriented outputs without rerunning Random Projection or RanPAC. It focuses on the five RP methods under the strongest feature and dimension setting, and it also visualizes the interaction between RP method and projection dimension for the embedding feature setting.

In [79]:
RP_DISPLAY_NAMES = {
    "gauss": "Gaussian RP",
    "sparse": "Sparse RP",
    "sign": "Sign RP",
    "srht": "SRHT",
    "bg": "Bernoulli-Gaussian RP",
}
RP_PLOT_ORDER = ["gauss", "sparse", "sign", "srht", "bg"]


def method_display_name(rp_alias):
    return RP_DISPLAY_NAMES.get(str(rp_alias), str(rp_alias))


def rp_order_index(rp_alias):
    try:
        return RP_PLOT_ORDER.index(str(rp_alias))
    except ValueError:
        return len(RP_PLOT_ORDER)


# Controlled table: same feature representation and same projection dimension.
rp_controlled_emb_d500 = mean_std_v6[
    (mean_std_v6["feature_alias"].astype(str) == "emb")
    & (mean_std_v6["projection_dim"].astype(int) == 500)
].copy()

require(len(rp_controlled_emb_d500) == 5, f"Expected 5 controlled RP rows, got {len(rp_controlled_emb_d500)}")
rp_controlled_emb_d500["rp_method_display"] = rp_controlled_emb_d500["rp_alias"].map(method_display_name)
rp_controlled_emb_d500["method_order"] = rp_controlled_emb_d500["rp_alias"].map(rp_order_index)

rp_controlled_emb_d500 = rp_controlled_emb_d500.sort_values(
    ["mean_enrollment_aware_accuracy", "mean_enrollment_aware_eer"],
    ascending=[False, True],
).reset_index(drop=True)

rp_controlled_emb_d500["controlled_rank_by_accuracy"] = np.arange(1, len(rp_controlled_emb_d500) + 1)

rp_controlled_columns = [
    "controlled_rank_by_accuracy",
    "feature_setting_full",
    "feature_alias",
    "rp_method_full",
    "rp_alias",
    "rp_method_display",
    "projection_dim",
    "mean_enrollment_aware_accuracy",
    "std_enrollment_aware_accuracy",
    "mean_enrollment_aware_macro_f1",
    "std_enrollment_aware_macro_f1",
    "mean_enrollment_aware_eer",
    "std_enrollment_aware_eer",
    "mean_runtime_seconds",
    "std_runtime_seconds",
    "num_seeds",
]

CONTROLLED_RP_TABLE_PATH = TABLE_DIR / "rp_controlled_emb_d500.csv"
rp_controlled_emb_d500[rp_controlled_columns].to_csv(
    CONTROLLED_RP_TABLE_PATH,
    index=False,
    encoding="utf-8-sig"
)


def blend_color(base_color, strength):
    strength = float(np.clip(strength, 0.0, 1.0))
    return tuple(
        int(COLORS["white"][i] * (1.0 - strength) + base_color[i] * strength)
        for i in range(3)
    )


def save_controlled_rp_dual_bar(table, output_path):
    plot_table = table.sort_values("controlled_rank_by_accuracy").copy()
    labels = plot_table["rp_method_display"].astype(str).tolist()
    acc_values = plot_table["mean_enrollment_aware_accuracy"].astype(float).to_numpy()
    eer_values = plot_table["mean_enrollment_aware_eer"].astype(float).to_numpy()

    width, height = 1400, 900
    plot_left, plot_right, plot_top, plot_bottom = 360, width - 80, 130, height - 105
    plot_width = plot_right - plot_left
    plot_height = plot_bottom - plot_top

    image = Image.new("RGB", (width, height), COLORS["white"])
    draw = ImageDraw.Draw(image)

    draw_centered(
        draw,
        (width / 2, 30),
        "Five RP Methods under Embedding and 500 Dimensions",
        FONT_TITLE_CLEAR,
        COLORS["dark"]
    )
    draw_centered(
        draw,
        ((plot_left + plot_right) / 2, height - 60),
        "Metric value",
        FONT_AXIS_CLEAR,
        COLORS["dark"]
    )

    # Legend is placed above the plot area to avoid overlap with bars.
    legend_x, legend_y = plot_left, 88
    draw.rectangle((legend_x, legend_y, legend_x + 20, legend_y + 20), fill=COLORS["blue"])
    draw.text((legend_x + 30, legend_y - 2), "Mean accuracy", font=FONT_LABEL_CLEAR, fill=COLORS["dark"])

    legend_x2 = legend_x + 235
    draw.rectangle((legend_x2, legend_y, legend_x2 + 20, legend_y + 20), fill=COLORS["orange"])
    draw.text((legend_x2 + 30, legend_y - 2), "Mean EER (lower is better)", font=FONT_LABEL_CLEAR, fill=COLORS["dark"])

    x_min, x_max = 0.0, 1.0
    for i in range(6):
        frac = i / 5
        x = plot_left + frac * plot_width
        draw.line((x, plot_top, x, plot_bottom), fill=COLORS["light_gray"], width=1)
        draw.text((x - 16, plot_bottom + 12), f"{frac:.1f}", font=FONT_TICK_CLEAR, fill=COLORS["gray"])

    draw.rectangle((plot_left, plot_top, plot_right, plot_bottom), outline=COLORS["dark"], width=2)

    row_height = plot_height / max(len(plot_table), 1)
    for idx, (label, acc, eer) in enumerate(zip(labels, acc_values, eer_values)):
        y_center = plot_top + row_height * (idx + 0.5)
        label_text_value = f"{idx + 1}. {label}"
        label_box = draw.textbbox((0, 0), label_text_value, font=FONT_LABEL_CLEAR)
        label_h = label_box[3] - label_box[1]
        draw.text((28, y_center - label_h / 2), label_text_value, font=FONT_LABEL_CLEAR, fill=COLORS["dark"])

        acc_len = (acc - x_min) / (x_max - x_min) * plot_width
        eer_len = (eer - x_min) / (x_max - x_min) * plot_width

        bar_h = 20
        gap = 9
        acc_y0, acc_y1 = y_center - gap - bar_h, y_center - gap
        eer_y0, eer_y1 = y_center + gap, y_center + gap + bar_h

        draw.rectangle((plot_left, acc_y0, plot_left + acc_len, acc_y1), fill=COLORS["blue"])
        draw.rectangle((plot_left, eer_y0, plot_left + eer_len, eer_y1), fill=COLORS["orange"])

        draw.text(
            (min(plot_left + acc_len + 9, plot_right - 95), acc_y0 - 2),
            f"acc {acc:.3f}",
            font=FONT_TICK_CLEAR,
            fill=COLORS["dark"],
        )
        draw.text(
            (min(plot_left + eer_len + 9, plot_right - 95), eer_y0 - 2),
            f"eer {eer:.3f}",
            font=FONT_TICK_CLEAR,
            fill=COLORS["dark"],
        )

    image.save(output_path)


def save_rp_dim_heatmap_emb(data, output_path):
    emb_data = data[data["feature_alias"].astype(str) == "emb"].copy()

    dims = [100, 200, 500]
    methods = [alias for alias in RP_PLOT_ORDER if alias in set(emb_data["rp_alias"].astype(str))]
    require(len(methods) == 5, f"Expected 5 RP methods in embedding heatmap, got {len(methods)}")

    def value_lookup(metric, rp_alias, dim):
        matched = emb_data[
            (emb_data["rp_alias"].astype(str) == rp_alias)
            & (emb_data["projection_dim"].astype(int) == int(dim))
        ]
        require(len(matched) == 1, f"Missing heatmap value for {rp_alias}, dim {dim}")
        return float(matched.iloc[0][metric])

    acc_values = np.array([
        [value_lookup("mean_enrollment_aware_accuracy", m, d) for d in dims]
        for m in methods
    ])
    eer_values = np.array([
        [value_lookup("mean_enrollment_aware_eer", m, d) for d in dims]
        for m in methods
    ])

    width, height = 1320, 780
    cell_w, cell_h = 130, 66
    top = 170

    # Increase spacing and use row labels only once on the left panel
    left1 = 170
    left2 = 650

    image = Image.new("RGB", (width, height), COLORS["white"])
    draw = ImageDraw.Draw(image)

    draw_centered(
        draw,
        (width / 2, 28),
        "RP Method x Projection Dimension under Embedding Features",
        FONT_TITLE,
        COLORS["dark"]
    )

    draw.text((left1 + 130, 94), "Mean accuracy", font=FONT_AXIS, fill=COLORS["dark"])
    draw.text((left2 + 135, 94), "Mean EER lower is better", font=FONT_AXIS, fill=COLORS["dark"])

    def draw_heatmap(matrix, left, base_color, lower_is_better=False, show_row_labels=True):
        values = matrix.astype(float)
        v_min, v_max = float(np.min(values)), float(np.max(values))
        if v_max == v_min:
            v_max = v_min + 1e-6

        # Column labels
        for j, dim in enumerate(dims):
            x = left + (j + 1) * cell_w
            draw_centered(draw, (x + cell_w / 2, top - 35), str(dim), FONT_SMALL, COLORS["dark"])

        # Row labels and cells
        for i, method in enumerate(methods):
            y = top + i * cell_h

            if show_row_labels:
                draw.text((left - 70, y + 20), method_display_name(method), font=FONT_TINY, fill=COLORS["dark"])

            for j, dim in enumerate(dims):
                value = values[i, j]
                norm = (value - v_min) / (v_max - v_min)
                if lower_is_better:
                    norm = 1.0 - norm

                fill = blend_color(base_color, 0.18 + 0.72 * norm)

                x0 = left + (j + 1) * cell_w
                y0 = y

                draw.rectangle(
                    (x0, y0, x0 + cell_w, y0 + cell_h),
                    fill=fill,
                    outline=COLORS["white"],
                    width=2
                )
                draw_centered(
                    draw,
                    (x0 + cell_w / 2, y0 + 20),
                    f"{value:.3f}",
                    FONT_TINY,
                    COLORS["dark"]
                )

        draw.rectangle(
            (left + cell_w, top, left + cell_w * (len(dims) + 1), top + cell_h * len(methods)),
            outline=COLORS["dark"],
            width=2
        )

    draw_heatmap(
        acc_values,
        left1,
        COLORS["blue"],
        lower_is_better=False,
        show_row_labels=True
    )

    draw_heatmap(
        eer_values,
        left2,
        COLORS["orange"],
        lower_is_better=True,
        show_row_labels=False
    )

    draw_centered(
        draw,
        (left1 + cell_w * 2.5, top + cell_h * len(methods) + 30),
        "Projection dimension",
        FONT_SMALL,
        COLORS["dark"]
    )
    draw_centered(
        draw,
        (left2 + cell_w * 2.5, top + cell_h * len(methods) + 30),
        "Projection dimension",
        FONT_SMALL,
        COLORS["dark"]
    )

    draw.text(
        (100, height - 200),
        "Darker cells indicate better values within each panel. For EER, lower values are better.",
        font=FONT_SMALL,
        fill=COLORS["gray"]
    )

    image.save(output_path)


CONTROLLED_RP_FIG_PATH = FIGURE_DIR / "rp_controlled_emb_d500_acc_eer.png"
RP_DIM_HEATMAP_PATH = FIGURE_DIR / "rp_dim_heatmap_emb_acc_eer.png"

save_controlled_rp_dual_bar(rp_controlled_emb_d500, CONTROLLED_RP_FIG_PATH)
save_rp_dim_heatmap_emb(mean_std_v6, RP_DIM_HEATMAP_PATH)

print("Controlled RP table saved =", CONTROLLED_RP_TABLE_PATH.exists())
print("Controlled RP figure saved =", CONTROLLED_RP_FIG_PATH.exists())
print("RP x dimension heatmap saved =", RP_DIM_HEATMAP_PATH.exists())


Controlled RP table saved = True
Controlled RP figure saved = True
RP x dimension heatmap saved = True


## 9. Online stability curve for Top 5 balanced settings

For each Top 5 balanced setting, the three seed prediction logs are averaged after downsampling every 100 samples.


In [82]:
balanced_top5 = pd.read_csv(TABLE_DIR / "top10_balanced.csv").head(5)
stability_curves = []

for _, setting in balanced_top5.iterrows():
    rows = final_df[
        (final_df["feature_setting_full"] == setting["feature_setting_full"])
        & (final_df["rp_method_full"] == setting["rp_method_full"])
        & (final_df["projection_dim"] == int(setting["projection_dim"]))
    ].sort_values("seed")
    seed_curves = []
    sample_index_ref = None
    for _, seed_row in rows.iterrows():
        log_df = pd.read_csv(
            seed_row["prediction_log_path"],
            usecols=["sample_index", "running_enrollment_aware_accuracy"],
        )
        log_df = log_df.iloc[::STABILITY_DOWNSAMPLE_STEP].copy()
        if sample_index_ref is None:
            sample_index_ref = log_df["sample_index"].to_numpy()
        seed_curves.append(log_df["running_enrollment_aware_accuracy"].to_numpy(dtype=float))
    stacked = np.vstack(seed_curves)
    valid_counts = np.sum(np.isfinite(stacked), axis=0)
    mean_curve = np.full(stacked.shape[1], np.nan, dtype=float)
    np.divide(np.nansum(stacked, axis=0), valid_counts, out=mean_curve, where=valid_counts > 0)
    stability_curves.append({
        "label": f"{setting['feature_alias']}_{setting['rp_alias']}_d{int(setting['projection_dim'])}",
        "x": sample_index_ref,
        "y": mean_curve,
    })

save_multi_line(
    stability_curves,
    FIGURE_DIR / "stability_top5.png",
    "Top 5 Balanced Settings: Online Stability",
    "Sample index",
    "Mean running enrollment-aware accuracy",
)

print("Stability figure exists =", (FIGURE_DIR / "stability_top5.png").exists())


Stability figure exists = True


## 10. Additional online stability curve for five RP methods

This figure compares the online stability of the five RP methods under the same feature representation and projection dimension. It is designed for the thesis discussion of RP-specific online learning behaviour.

In [85]:

# Five-RP online stability under the strongest controlled feature and dimension setting.
rp_stability_curves = []

for rp_alias in RP_PLOT_ORDER:
    rows = final_df[
        (final_df["feature_alias"].astype(str) == "emb")
        & (final_df["projection_dim"].astype(int) == 500)
        & (final_df["rp_alias"].astype(str) == rp_alias)
    ].sort_values("seed")
    require(len(rows) == EXPECTED_NUM_SEEDS, f"Expected 3 seeds for emb {rp_alias} d500, got {len(rows)}")

    seed_curves = []
    sample_index_ref = None
    for _, seed_row in rows.iterrows():
        log_df = pd.read_csv(
            seed_row["prediction_log_path"],
            usecols=["sample_index", "running_enrollment_aware_accuracy"],
        )
        log_df = log_df.iloc[::STABILITY_DOWNSAMPLE_STEP].copy()
        if sample_index_ref is None:
            sample_index_ref = log_df["sample_index"].to_numpy()
        seed_curves.append(log_df["running_enrollment_aware_accuracy"].to_numpy(dtype=float))

    stacked = np.vstack(seed_curves)
    valid_counts = np.sum(np.isfinite(stacked), axis=0)
    mean_curve = np.full(stacked.shape[1], np.nan, dtype=float)
    np.divide(np.nansum(stacked, axis=0), valid_counts, out=mean_curve, where=valid_counts > 0)
    rp_stability_curves.append({
        "label": method_display_name(rp_alias),
        "x": sample_index_ref,
        "y": mean_curve,
    })

RP_STABILITY_FIG_PATH = FIGURE_DIR / "rp_stability_emb_d500.png"
save_multi_line(
    rp_stability_curves,
    RP_STABILITY_FIG_PATH,
    "Five RP: Online Stability(Embedding + 500)",
    "Sample index",
    "Mean running enrollment-aware accuracy",
)

print("Five-RP stability figure saved =", RP_STABILITY_FIG_PATH.exists())


Five-RP stability figure saved = True


## 11. EER curves for Top 5 balanced settings

For each Top 5 balanced setting, the representative curve uses the seed with the lowest enrollment-aware EER.


In [88]:
eer_curve_output_paths = []
representative_curve_rows = []

for _, setting in balanced_top5.iterrows():
    rows = final_df[
        (final_df["feature_setting_full"] == setting["feature_setting_full"])
        & (final_df["rp_method_full"] == setting["rp_method_full"])
        & (final_df["projection_dim"] == int(setting["projection_dim"]))
    ].sort_values("enrollment_aware_eer")
    representative = rows.iloc[0]
    score_data = np.load(representative["score_output_path"], allow_pickle=False)
    true_scores = score_data["true_scores"].astype(float)
    max_non_true_scores = score_data["max_non_true_scores"].astype(float)
    enrollment_flags = score_data["enrollment_aware_counted_flags"].astype(bool)
    valid_mask = np.isfinite(true_scores) & np.isfinite(max_non_true_scores) & enrollment_flags
    curve_result = compute_eer_curve(true_scores[valid_mask], max_non_true_scores[valid_mask])
    curve = curve_result["curve"]

    base = f"eer_curve_{clean_name(representative['feature_alias'])}_{clean_name(representative['rp_alias'])}_d{int(representative['projection_dim'])}_{clean_name(representative['seed_label'])}"
    csv_path = EER_CURVE_DIR / f"{base}.csv"
    png_path = EER_CURVE_DIR / f"{base}.png"
    curve.to_csv(csv_path, index=False, encoding="utf-8-sig")

    save_multi_line(
        [
            {"label": "FAR", "x": curve["threshold"].to_numpy(), "y": curve["FAR"].to_numpy()},
            {"label": "FRR", "x": curve["threshold"].to_numpy(), "y": curve["FRR"].to_numpy()},
        ],
        png_path,
        f"EER Curve: {base}",
        "Threshold",
        "Error rate",
        y_min=0.0,
        y_max=1.0,
    )

    eer_curve_output_paths.extend([csv_path, png_path])
    representative_curve_rows.append({
        "setting": f"{representative['feature_alias']}_{representative['rp_alias']}_d{int(representative['projection_dim'])}",
        "representative_seed": representative["seed_label"],
        "representative_enrollment_aware_eer": float(representative["enrollment_aware_eer"]),
        "curve_csv_path": str(csv_path),
        "curve_png_path": str(png_path),
    })

representative_curve_df = pd.DataFrame(representative_curve_rows)
representative_curve_df.to_csv(EER_CURVE_DIR / "top5_eer_curve_representatives.csv", index=False, encoding="utf-8-sig")
print("EER curves saved =", all(path.exists() for path in eer_curve_output_paths))
print(representative_curve_df.to_string(index=False))


EER curves saved = True
        setting representative_seed  representative_enrollment_aware_eer                                                      curve_csv_path                                                      curve_png_path
    emb_bg_d500                 s82                             0.213630     F:\ECG\results\eval_v6\eer_curves\eer_curve_emb_bg_d500_s82.csv     F:\ECG\results\eval_v6\eer_curves\eer_curve_emb_bg_d500_s82.png
 emb_gauss_d500                 s27                             0.214514  F:\ECG\results\eval_v6\eer_curves\eer_curve_emb_gauss_d500_s27.csv  F:\ECG\results\eval_v6\eer_curves\eer_curve_emb_gauss_d500_s27.png
emb_sparse_d500                 s42                             0.214257 F:\ECG\results\eval_v6\eer_curves\eer_curve_emb_sparse_d500_s42.csv F:\ECG\results\eval_v6\eer_curves\eer_curve_emb_sparse_d500_s42.png
  emb_sign_d500                 s82                             0.251910   F:\ECG\results\eval_v6\eer_curves\eer_curve_emb_sign_d500_s82.csv

## 12. Final V6 report

The report is written in Chinese with English technical terms where useful.


In [90]:
def markdown_table(dataframe, columns, float_digits=6):
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, record in dataframe[columns].iterrows():
        values = []
        for value in record.tolist():
            if isinstance(value, (float, np.floating)):
                values.append(f"{float(value):.{float_digits}f}")
            else:
                values.append(str(value))
        rows.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator] + rows)


best_by_accuracy = mean_std_v6.sort_values(
    ["mean_enrollment_aware_accuracy", "mean_enrollment_aware_macro_f1", "mean_enrollment_aware_eer"],
    ascending=[False, False, True],
).iloc[0]
best_by_eer = mean_std_v6.sort_values(
    ["mean_enrollment_aware_eer", "mean_enrollment_aware_accuracy", "mean_enrollment_aware_macro_f1"],
    ascending=[True, False, False],
).iloc[0]
best_balanced = mean_std_v6.sort_values(
    ["overall_rank_score", "rank_by_mean_accuracy", "rank_by_mean_eer"],
    ascending=[True, True, True],
).iloc[0]

feature_best = by_feature.sort_values("mean_enrollment_aware_accuracy_mean", ascending=False).iloc[0]
method_best = by_method.sort_values("mean_enrollment_aware_accuracy_mean", ascending=False).iloc[0]
dim_best = by_dim.sort_values("mean_enrollment_aware_accuracy_mean", ascending=False).iloc[0]
dim_runtime = by_dim.sort_values("mean_runtime_seconds_mean", ascending=True)

dim_efficiency = by_dim[["projection_dim", "mean_enrollment_aware_accuracy_mean", "mean_enrollment_aware_eer_mean", "mean_runtime_seconds_mean"]].copy()
dim_efficiency["estimated_template_size_mb"] = 36103 * dim_efficiency["projection_dim"].astype(float) * 4 / (1024 ** 2)

report_lines = [
    "# V6 Step 8 Final Evaluation Report",
    "",
    "## 1. Purpose of V6",
    "",
    "The purpose of V6 is to extend the V5 single-seed results into publication-ready multi-seed results. V6 uses three random projection seeds to evaluate RP method, projection dimension, and feature setting stability.",
    "",
    "## 2. Difference Between V6 and V5",
    "",
    "V5 used seed27 for 45 RP + RanPAC experiments. V6 uses seed27, seed42, and seed82 for 135 RP-only RanPAC experiments and reports mean +/- std. V6 Step 8 evaluates existing RP and RanPAC outputs without rerunning Step 7.",
    "",
    "## 3. Why V6 Is RP-only",
    "",
    "Random Projection is treated as the cancellable biometric transformation. The No-RP baseline is not a cancellable transformed template, so it is excluded from the V6 RP-only analysis. V6 Step 8 summarizes 135 RP-only rows.",
    "",
    "## 4. Multi-seed Setting",
    "",
    "V6 uses three seeds: 27, 42, and 82. Each feature/method/dimension setting has three seed runs, and mean_std_v6.csv records num_seeds = 3.",
    "",
    "## 5. EER Calculation Rule",
    "",
    "EER uses a score-based verification interpretation: genuine score = true_score and impostor score = max_non_true_score. NaN/Inf scores are filtered; enrollment-aware EER uses enrollment_aware_counted_flags = True and is reported as enrollment_aware_eer.",
    "",
    "## 6. Best by mean accuracy",
    "",
    f"- Setting: {label_text(best_by_accuracy)}",
    f"- Mean enrollment-aware accuracy: {best_by_accuracy['mean_enrollment_aware_accuracy']:.6f}",
    f"- Mean macro-F1: {best_by_accuracy['mean_enrollment_aware_macro_f1']:.6f}",
    f"- Mean EER: {best_by_accuracy['mean_enrollment_aware_eer']:.6f}",
    "",
    "## 7. Best by mean EER",
    "",
    f"- Setting: {label_text(best_by_eer)}",
    f"- Mean EER: {best_by_eer['mean_enrollment_aware_eer']:.6f}",
    f"- Mean enrollment-aware accuracy: {best_by_eer['mean_enrollment_aware_accuracy']:.6f}",
    f"- Mean macro-F1: {best_by_eer['mean_enrollment_aware_macro_f1']:.6f}",
    "",
    "## 8. Best balanced setting",
    "",
    f"- Setting: {label_text(best_balanced)}",
    f"- Overall rank score: {int(best_balanced['overall_rank_score'])}",
    f"- Accuracy rank: {int(best_balanced['rank_by_mean_accuracy'])}",
    f"- Macro-F1 rank: {int(best_balanced['rank_by_mean_macro_f1'])}",
    f"- EER rank: {int(best_balanced['rank_by_mean_eer'])}",
    "",
    "## 9. Feature setting comparison",
    "",
    markdown_table(by_feature.sort_values("mean_enrollment_aware_accuracy_mean", ascending=False), [
        "feature_setting_full",
        "mean_enrollment_aware_accuracy_mean",
        "mean_enrollment_aware_macro_f1_mean",
        "mean_enrollment_aware_eer_mean",
        "mean_runtime_seconds_mean",
    ]),
    "",
    f"Feature-level results show,{feature_best['feature_setting_full']} has the highest mean enrollment-aware accuracy.",
    "",
    "## 10. RP method comparison",
    "",
    markdown_table(by_method.sort_values("mean_enrollment_aware_accuracy_mean", ascending=False), [
        "rp_method_full",
        "mean_enrollment_aware_accuracy_mean",
        "mean_enrollment_aware_macro_f1_mean",
        "mean_enrollment_aware_eer_mean",
        "mean_runtime_seconds_mean",
    ]),
    "",
    f"Method-level results show,{method_best['rp_method_full']} has the highest mean enrollment-aware accuracy.",
    "",
    "## 11. Projection dimension comparison",
    "",
    markdown_table(by_dim.sort_values("projection_dim"), [
        "projection_dim",
        "mean_enrollment_aware_accuracy_mean",
        "mean_enrollment_aware_macro_f1_mean",
        "mean_enrollment_aware_eer_mean",
        "mean_runtime_seconds_mean",
    ]),
    "",
    f"Dimension-level results show,dim{int(dim_best['projection_dim'])} has the highest mean enrollment-aware accuracy.",
    "",
    "## 12. Seed stability discussion",
    "",
    "Each V6 setting has three seeds. The std columns in mean_std_v6.csv describe sensitivity to RP method seed. Top settings with lower std indicate more stable method behavior across seeds.",
    "",
    "## 13. Online stability curve discussion",
    "",
    "stability_top5.png shows the Top 5 balanced settings across three seed prediction logs, downsampled every 100 samples to show mean running enrollment-aware accuracy. It summarizes the streaming online learning behavior of RanPAC.",
    "",
    "## 14. Performance-efficiency trade-off discussion",
    "",
    markdown_table(dim_efficiency.sort_values("projection_dim"), [
        "projection_dim",
        "mean_enrollment_aware_accuracy_mean",
        "mean_enrollment_aware_eer_mean",
        "mean_runtime_seconds_mean",
        "estimated_template_size_mb",
    ]),
    "",
    "Projection dimension affects runtime and template storage. dim500 is the main V6 trade-off point; dim100 and dim200 are efficiency-oriented settings.",
    "",
    "## 15. Thesis writing notes",
    "",
    "- Scope: Random Projection-based cancellable transformations under the same RanPAC online protocol.",
    "- RanPAC is fixed as the online learning evaluator.",
    "- No-RP is excluded because it is not a cancellable transformed template.",
    "- Metrics include enrollment-aware accuracy, macro-F1, and enrollment-aware EER.",
    "",
    "## 16. Limitations",
    "",
    "- V6 uses UofTDB sit-only 5-second segments.",
    "- V6 does not include dim1000; dim1000 is left for future extended analysis.",
    "- EER adapts multi-class identification scores into a verification-style interpretation.",
    "- No-RP baseline rows are generated by Step 6 but excluded here.",
    "",
    "## Output paths",
    "",
    f"- Final per-experiment table: `{FINAL_TABLE_PATH}`",
    f"- Mean/std table: `{MEAN_STD_PATH}`",
    f"- Figures: `{FIGURE_DIR}`",
    f"- EER curves: `{EER_CURVE_DIR}`",
    f"- Report: `{REPORT_PATH}`",
]

REPORT_PATH.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("Final report exists =", REPORT_PATH.exists())
print("Best by mean accuracy =", label_text(best_by_accuracy))
print("Best by mean EER =", label_text(best_by_eer))
print("Best balanced setting =", label_text(best_balanced))


Final report exists = True
Best by mean accuracy = emb_bg_d500
Best by mean EER = emb_bg_d500
Best balanced setting = emb_bg_d500


## 13. Final validation

The final printout follows AGENT.md section 27.14 exactly.


In [94]:
ranking_table_paths = [
    TABLE_DIR / "top10_acc.csv",
    TABLE_DIR / "top10_f1.csv",
    TABLE_DIR / "top10_eer.csv",
    TABLE_DIR / "top10_balanced.csv",
]
group_table_paths = [
    TABLE_DIR / "by_feature.csv",
    TABLE_DIR / "by_method.csv",
    TABLE_DIR / "by_dim.csv",
]
required_figure_paths = [
    FIGURE_DIR / "acc_by_feature.png",
    FIGURE_DIR / "acc_by_method.png",
    FIGURE_DIR / "acc_by_dim.png",
    FIGURE_DIR / "eer_by_feature.png",
    FIGURE_DIR / "eer_by_method.png",
    FIGURE_DIR / "eer_by_dim.png",
    FIGURE_DIR / "acc_eer_scatter.png",
    FIGURE_DIR / "top10_acc_eer.png",
    FIGURE_DIR / "acc_dim_std.png",
    FIGURE_DIR / "eer_dim_std.png",
    FIGURE_DIR / "stability_top5.png",
    FIGURE_DIR / "rp_controlled_emb_d500_acc_eer.png",
    FIGURE_DIR / "rp_dim_heatmap_emb_acc_eer.png",
    FIGURE_DIR / "rp_stability_emb_d500.png",
]
rp_focused_table_paths = [
    TABLE_DIR / "rp_controlled_emb_d500.csv",
]
eer_curve_csv_count = len(list(EER_CURVE_DIR.glob("eer_curve_*.csv")))
eer_curve_png_count = len(list(EER_CURVE_DIR.glob("eer_curve_*.png")))

final_validation = {
    "Input Step 7 rows": input_step7_rows,
    "Final per-experiment rows": len(final_df),
    "Mean/std rows": len(mean_std_v6),
    "EER computed for all 135": bool(final_df[["strict_valid_eer", "enrollment_aware_eer"]].notna().all().all()),
    "Every mean/std row has num_seeds = 3": bool((mean_std_v6["num_seeds"] == 3).all()),
    "Ranking tables exist": all(path.exists() for path in ranking_table_paths),
    "Group tables exist": all(path.exists() for path in group_table_paths),
    "Figures exist": all(path.exists() for path in required_figure_paths),
    "RP-focused thesis table exists": all(path.exists() for path in rp_focused_table_paths),
    "RP-focused thesis figures exist": all(path.exists() for path in required_figure_paths[-3:]),
    "Stability figure exists": (FIGURE_DIR / "stability_top5.png").exists(),
    "EER curves saved": eer_curve_csv_count >= 5 and eer_curve_png_count >= 5,
    "Final report exists": REPORT_PATH.exists(),
    "No no-RP rows included": bool(no_norp_input and (final_df["rp_alias"].astype(str) != "norp").all()),
}

final_validation_text = "\n".join([
    f"Input Step 7 rows = {final_validation['Input Step 7 rows']}",
    f"Final per-experiment rows = {final_validation['Final per-experiment rows']}",
    f"Mean/std rows = {final_validation['Mean/std rows']}",
    f"EER computed for all 135 = {final_validation['EER computed for all 135']}",
    f"Every mean/std row has num_seeds = 3 = {final_validation['Every mean/std row has num_seeds = 3']}",
    f"Ranking tables exist = {final_validation['Ranking tables exist']}",
    f"Group tables exist = {final_validation['Group tables exist']}",
    f"Figures exist = {final_validation['Figures exist']}",
    f"Stability figure exists = {final_validation['Stability figure exists']}",
    f"EER curves saved = {final_validation['EER curves saved']}",
    f"Final report exists = {final_validation['Final report exists']}",
    f"No no-RP rows included = {final_validation['No no-RP rows included']}",
])

with open(OUTPUT_DIR / "eval_v6_validation_report.json", "w", encoding="utf-8") as f:
    json.dump(final_validation, f, indent=2, ensure_ascii=False)

print(final_validation_text)


Input Step 7 rows = 135
Final per-experiment rows = 135
Mean/std rows = 45
EER computed for all 135 = True
Every mean/std row has num_seeds = 3 = True
Ranking tables exist = True
Group tables exist = True
Figures exist = True
Stability figure exists = True
EER curves saved = True
Final report exists = True
No no-RP rows included = True
